# Notebook 4: Ablation Study

Reproduces Table 6 (Ablation) of the paper.

**Configurations tested (ObliQA, n=200):**
- BGE only, BM25 only
- BGE + CE (no window)
- BGE + CE + Window
- Full pipeline
- Rank by RRF not CE
- Rank by window max
- Fixed top-k=1, 5, 10
- Full pipeline (n=100)

**Window ablation (RegulationQA, n=1,002):**
- Full pipeline with vs without window (span-level F1)

**Runtime:** ~90 min on T4

In [ ]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 nltk

import pickle, json, string, requests, time, numpy as np, pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from nltk.tokenize import sent_tokenize
import faiss, nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
# Load ObliQA
print('Loading ObliQA...')
test_data = json.loads(requests.get(
    'https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/ObliQA_test.json'
).text)
all_passages = []
for doc_id in range(1, 41):
    r = requests.get(f'https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/StructuredRegulatoryDocuments/{doc_id}.json')
    if r.status_code == 200:
        all_passages.extend(json.loads(r.text))
    time.sleep(0.2)
clean_passages = []
for p in all_passages:
    text = p['Passage'].strip()
    if len(text) < 30:
        continue
    if len(text) > 1500:
        start = 0
        while start < len(text):
            chunk = text[start:start+1000]
            if len(chunk) < 100 and start > 0:
                break
            clean_passages.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'], 'Passage': chunk})
            start += 800
    else:
        clean_passages.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'], 'Passage': text})

bge_model     = SentenceTransformer('BAAI/bge-base-en-v1.5')
embeddings    = bge_model.encode([p['Passage'] for p in clean_passages], batch_size=128, show_progress_bar=True, normalize_embeddings=True)
obliqua_index = faiss.IndexFlatIP(embeddings.shape[1])
obliqua_index.add(embeddings.astype(np.float32))
corpus        = [p['Passage'].lower().translate(str.maketrans('', '', string.punctuation)).split() for p in clean_passages]
obliqua_bm25  = BM25Okapi(corpus)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

pid_to_idx = {}
for i, p in enumerate(clean_passages):
    pid = (p['DocumentID'], p['PassageID'])
    if pid not in pid_to_idx:
        pid_to_idx[pid] = i

print(f'ObliQA ready. Passages: {len(clean_passages)}')

In [ ]:
# Load RegulationQA
print('Loading RegulationQA...')
with open('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/chunks.pkl', 'rb') as f:
    rqa_chunks = pickle.load(f)
rqa_index = faiss.read_index('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/faiss_index.bin')
with open('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/bm25.pkl', 'rb') as f:
    rqa_bm25 = pickle.load(f)
rqa_df = pd.read_csv('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/regulationqa_v1.csv')
print(f'RegulationQA ready. Chunks: {len(rqa_chunks)}, Questions: {len(rqa_df)}')

In [ ]:
# Helper functions
def tokenize(text):
    return text.lower().translate(str.maketrans('', '', string.punctuation)).split()

def inner_decay(sentences, ce_scores, threshold=0.0, delta=2.0):
    if not sentences:
        return []
    scored = sorted(zip(ce_scores, sentences), reverse=True)
    selected = []
    for i, (score, sent) in enumerate(scored):
        if score < threshold:
            break
        if i > 0 and (scored[i-1][0] - score) > delta:
            break
        selected.append((score, sent))
    return selected if selected else [scored[0]]

def get_window_obliqua(doc_id, passage_idx):
    doc_passages = [(i, p) for i, p in enumerate(clean_passages) if p['DocumentID'] == doc_id]
    current_pos  = next((pos for pos, (i, p) in enumerate(doc_passages) if i == passage_idx), None)
    if current_pos is None:
        return clean_passages[passage_idx]['Passage']
    window = []
    for offset in [-1, 0, 1]:
        pos = current_pos + offset
        if 0 <= pos < len(doc_passages):
            window.append(doc_passages[pos][1]['Passage'])
    return ' '.join(window)

def get_window_rqa(doc_name, passage_idx):
    doc_chunks  = [(i, c) for i, c in enumerate(rqa_chunks) if c['doc_name'] == doc_name]
    current_pos = next((pos for pos, (i, c) in enumerate(doc_chunks) if i == passage_idx), None)
    if current_pos is None:
        return rqa_chunks[passage_idx]['text']
    window = []
    for offset in [-1, 0, 1]:
        pos = current_pos + offset
        if 0 <= pos < len(doc_chunks):
            window.append(doc_chunks[pos][1]['text'])
    return ' '.join(window)

def token_f1(pred, gt):
    pred_t = pred.lower().split()
    gt_t   = gt.lower().split()
    common = set(pred_t) & set(gt_t)
    if not common:
        return 0.0
    p = len(common) / len(pred_t)
    r = len(common) / len(gt_t)
    return 2 * p * r / (p + r)

print('Helpers ready.')

In [ ]:
# ObliQA evaluator
def evaluate_obliqua(test_data, retrieve_fn, top_n=10):
    recall_1, recall_5, recall_10, map_10 = [], [], [], []
    for item in tqdm(test_data):
        query   = item['Question']
        gt_pids = set((p['DocumentID'], p['PassageID']) for p in item['Passages'])
        results   = retrieve_fn(query, top_n=top_n)
        retrieved = [(r['DocumentID'], r['PassageID']) for r in results]
        recall_1.append(1.0 if any(p in gt_pids for p in retrieved[:1]) else 0.0)
        recall_5.append(1.0 if any(p in gt_pids for p in retrieved[:5]) else 0.0)
        recall_10.append(1.0 if any(p in gt_pids for p in retrieved[:10]) else 0.0)
        ap, hits = 0.0, 0
        for k, pid in enumerate(retrieved[:10], 1):
            if pid in gt_pids:
                hits += 1
                ap   += hits / k
        map_10.append(ap / max(len(gt_pids), 1))
    return {'Recall@1': np.mean(recall_1), 'Recall@5': np.mean(recall_5),
            'Recall@10': np.mean(recall_10), 'MAP@10': np.mean(map_10)}

# RegulationQA evaluator
def evaluate_rqa(df, retrieve_fn, top_n=100):
    f1_scores, r03, r05 = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        query        = row['question']
        ground_truth = str(row['ground_truth_answer'])
        results      = retrieve_fn(query, top_n=top_n)
        best_f1      = 0.0
        for r in results[:3]:
            for span in r.get('extracted_spans', []):
                f1 = token_f1(span, ground_truth)
                if f1 > best_f1:
                    best_f1 = f1
        f1_scores.append(best_f1)
        r03.append(1.0 if best_f1 >= 0.3 else 0.0)
        r05.append(1.0 if best_f1 >= 0.5 else 0.0)
    return {'Mean F1': np.mean(f1_scores), 'Recall@0.3': np.mean(r03), 'Recall@0.5': np.mean(r05)}

print('Evaluators ready.')

In [ ]:
# All ObliQA ablation retrieval functions
def retrieve_bge_only(query, top_n=10):
    qe = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = obliqua_index.search(qe, top_n)
    results, seen = [], set()
    for idx in indices[0]:
        p = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        if pid not in seen:
            seen.add(pid)
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID']})
    return results

def retrieve_bm25_only(query, top_n=10):
    scores  = obliqua_bm25.get_scores(tokenize(query))
    indices = np.argsort(scores)[::-1][:top_n]
    results, seen = [], set()
    for idx in indices:
        p = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        if pid not in seen:
            seen.add(pid)
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID']})
    return results

def _rrf_candidates(query, top_n=100, rrf_k=60):
    qe = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    _, bge_idx = obliqua_index.search(qe, top_n)
    bge_idx    = bge_idx[0]
    bm25_s     = obliqua_bm25.get_scores(tokenize(query))
    bm25_idx   = np.argsort(bm25_s)[::-1][:top_n]
    rrf = {}
    for rank, idx in enumerate(bge_idx):
        p = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        rrf[pid] = rrf.get(pid, 0) + 1/(rrf_k + rank + 1)
    for rank, idx in enumerate(bm25_idx):
        p = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        rrf[pid] = rrf.get(pid, 0) + 1/(rrf_k + rank + 1)
    return sorted(rrf.items(), key=lambda x: x[1], reverse=True), rrf

def retrieve_bge_ce_nowindow(query, top_n=100):
    qe = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = obliqua_index.search(qe, top_n)
    results, seen = [], set()
    for idx in indices[0]:
        p = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        if pid in seen:
            continue
        seen.add(pid)
        sents = sent_tokenize(p['Passage'])
        if not sents:
            continue
        ce_s = cross_encoder.predict([[query, s] for s in sents])
        kept = inner_decay(sents, ce_s)
        if kept:
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'],
                            'top_ce_score': float(max(ce_s))})
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

def retrieve_bge_ce_window(query, top_n=100):
    qe = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = obliqua_index.search(qe, top_n)
    results, seen = [], set()
    for rank, idx in enumerate(indices[0]):
        p = clean_passages[idx]
        pid = (p['DocumentID'], p['PassageID'])
        if pid in seen:
            continue
        seen.add(pid)
        curr_sents = sent_tokenize(p['Passage'])
        if not curr_sents:
            continue
        curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce  = float(max(curr_ce))
        win     = get_window_obliqua(p['DocumentID'], idx)
        win_s   = sent_tokenize(win)
        win_ce  = cross_encoder.predict([[query, s] for s in win_s])
        kept    = inner_decay(win_s, win_ce)
        if kept:
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'],
                            'top_ce_score': top_ce})
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

def retrieve_full_pipeline(query, top_n=100):
    sorted_pids, rrf = _rrf_candidates(query, top_n)
    results, seen = [], set()
    for pid, _ in sorted_pids[:50]:
        if pid in seen:
            continue
        seen.add(pid)
        idx = pid_to_idx.get(pid)
        if idx is None:
            continue
        p = clean_passages[idx]
        curr_sents = sent_tokenize(p['Passage'])
        if not curr_sents:
            continue
        curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce  = float(max(curr_ce))
        win     = get_window_obliqua(p['DocumentID'], idx)
        win_s   = sent_tokenize(win)
        win_ce  = cross_encoder.predict([[query, s] for s in win_s])
        kept    = inner_decay(win_s, win_ce)
        if kept:
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'],
                            'top_ce_score': top_ce})
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

def retrieve_rank_by_rrf(query, top_n=100):
    sorted_pids, rrf = _rrf_candidates(query, top_n)
    results, seen = [], set()
    for pid, rrf_score in sorted_pids[:50]:
        if pid in seen:
            continue
        seen.add(pid)
        idx = pid_to_idx.get(pid)
        if idx is None:
            continue
        p = clean_passages[idx]
        win   = get_window_obliqua(p['DocumentID'], idx)
        win_s = sent_tokenize(win)
        win_ce = cross_encoder.predict([[query, s] for s in win_s])
        kept  = inner_decay(win_s, win_ce)
        if kept:
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'],
                            'rrf_score': float(rrf_score)})
    results.sort(key=lambda x: x['rrf_score'], reverse=True)
    return results

def retrieve_rank_by_window_max(query, top_n=100):
    sorted_pids, rrf = _rrf_candidates(query, top_n)
    results, seen = [], set()
    for pid, _ in sorted_pids[:50]:
        if pid in seen:
            continue
        seen.add(pid)
        idx = pid_to_idx.get(pid)
        if idx is None:
            continue
        p = clean_passages[idx]
        win   = get_window_obliqua(p['DocumentID'], idx)
        win_s = sent_tokenize(win)
        win_ce = cross_encoder.predict([[query, s] for s in win_s])
        kept  = inner_decay(win_s, win_ce)
        if kept:
            results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'],
                            'top_ce_score': float(max(win_ce))})
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

def make_fixed_k(k):
    def retrieve_fixed_k(query, top_n=100):
        sorted_pids, rrf = _rrf_candidates(query, top_n)
        results, seen = [], set()
        for pid, _ in sorted_pids[:k]:
            if pid in seen:
                continue
            seen.add(pid)
            idx = pid_to_idx.get(pid)
            if idx is None:
                continue
            p = clean_passages[idx]
            curr_sents = sent_tokenize(p['Passage'])
            if not curr_sents:
                continue
            curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
            top_ce  = float(max(curr_ce))
            win     = get_window_obliqua(p['DocumentID'], idx)
            win_s   = sent_tokenize(win)
            win_ce  = cross_encoder.predict([[query, s] for s in win_s])
            kept    = inner_decay(win_s, win_ce)
            if kept:
                results.append({'DocumentID': p['DocumentID'], 'PassageID': p['PassageID'],
                                'top_ce_score': top_ce})
        results.sort(key=lambda x: x['top_ce_score'], reverse=True)
        return results
    return retrieve_fixed_k

print('All retrieval functions defined.')

In [ ]:
# Run ObliQA ablation (n=200 sample)
N = 200
ablation_results = {}

configs = [
    ('BGE only',          retrieve_bge_only,          10),
    ('BM25 only',         retrieve_bm25_only,          10),
    ('BGE + CE nowindow', retrieve_bge_ce_nowindow,   100),
    ('BGE + CE + Window', retrieve_bge_ce_window,     100),
    ('Full pipeline',     retrieve_full_pipeline,      100),
    ('Rank by RRF',       retrieve_rank_by_rrf,        100),
    ('Rank by win max',   retrieve_rank_by_window_max, 100),
    ('Fixed top-k=1',     make_fixed_k(1),             100),
    ('Fixed top-k=5',     make_fixed_k(5),             100),
    ('Fixed top-k=10',    make_fixed_k(10),            100),
]

for name, fn, top_n in configs:
    print(f'Running: {name}...')
    ablation_results[name] = evaluate_obliqua(test_data[:N], fn, top_n=top_n)
    m = ablation_results[name]
    print(f'  R@1={m["Recall@1"]:.3f} R@5={m["Recall@5"]:.3f} R@10={m["Recall@10"]:.3f} MAP={m["MAP@10"]:.3f}')

In [ ]:
# RegulationQA window ablation
rqa_pid_to_idx = {}
for i, c in enumerate(rqa_chunks):
    pid = (c['doc_name'], c['chunk_idx'])
    if pid not in rqa_pid_to_idx:
        rqa_pid_to_idx[pid] = i

def rqa_rrf_candidates(query, top_n=100, rrf_k=60):
    qe = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    _, bge_idx = rqa_index.search(qe, top_n)
    bge_idx    = bge_idx[0]
    bm25_s     = rqa_bm25.get_scores(tokenize(query))
    bm25_idx   = np.argsort(bm25_s)[::-1][:top_n]
    rrf = {}
    for rank, idx in enumerate(bge_idx):
        c = rqa_chunks[idx]
        pid = (c['doc_name'], c['chunk_idx'])
        rrf[pid] = rrf.get(pid, 0) + 1/(rrf_k + rank + 1)
    for rank, idx in enumerate(bm25_idx):
        c = rqa_chunks[idx]
        pid = (c['doc_name'], c['chunk_idx'])
        rrf[pid] = rrf.get(pid, 0) + 1/(rrf_k + rank + 1)
    sorted_pids = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:20]
    return sorted_pids

def rqa_retrieve_window(query, top_n=100):
    sorted_pids = rqa_rrf_candidates(query, top_n)
    results, seen = [], set()
    for pid, _ in sorted_pids:
        if pid in seen:
            continue
        seen.add(pid)
        idx = rqa_pid_to_idx.get(pid)
        if idx is None:
            continue
        c = rqa_chunks[idx]
        curr_sents = sent_tokenize(c['text'])
        if not curr_sents:
            continue
        curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce  = float(max(curr_ce))
        win     = get_window_rqa(c['doc_name'], idx)
        win_s   = sent_tokenize(win)
        win_ce  = cross_encoder.predict([[query, s] for s in win_s])
        kept    = inner_decay(win_s, win_ce)
        if kept:
            results.append({'doc_name': c['doc_name'], 'top_ce_score': top_ce,
                            'extracted_spans': [s for _, s in kept]})
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

def rqa_retrieve_nowindow(query, top_n=100):
    sorted_pids = rqa_rrf_candidates(query, top_n)
    results, seen = [], set()
    for pid, _ in sorted_pids:
        if pid in seen:
            continue
        seen.add(pid)
        idx = rqa_pid_to_idx.get(pid)
        if idx is None:
            continue
        c = rqa_chunks[idx]
        curr_sents = sent_tokenize(c['text'])
        if not curr_sents:
            continue
        curr_ce = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce  = float(max(curr_ce))
        kept    = inner_decay(curr_sents, curr_ce)
        if kept:
            results.append({'doc_name': c['doc_name'], 'top_ce_score': top_ce,
                            'extracted_spans': [s for _, s in kept]})
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

print('Running window ablation on RegulationQA (n=1002)...')
print('With window:')
r_window    = evaluate_rqa(rqa_df, rqa_retrieve_window)
print(f"  F1={r_window['Mean F1']:.4f} R@0.3={r_window['Recall@0.3']:.4f} R@0.5={r_window['Recall@0.5']:.4f}")
print('Without window:')
r_nowindow  = evaluate_rqa(rqa_df, rqa_retrieve_nowindow)
print(f"  F1={r_nowindow['Mean F1']:.4f} R@0.3={r_nowindow['Recall@0.3']:.4f} R@0.5={r_nowindow['Recall@0.5']:.4f}")
print(f"Delta F1: {r_window['Mean F1'] - r_nowindow['Mean F1']:+.4f}")

In [ ]:
# Print full ablation table
print(f'\n=== TABLE 6: Ablation Study ===')
print(f'{"Configuration":<25} {"R@1":>6} {"R@5":>6} {"R@10":>6} {"MAP":>6}')
print('=' * 50)
for name, metrics in ablation_results.items():
    print(f'{name:<25} {metrics["Recall@1"]:>6.2f} {metrics["Recall@5"]:>6.2f} '
          f'{metrics["Recall@10"]:>6.2f} {metrics["MAP@10"]:>6.3f}')
print(f'\nWindow ablation (RegulationQA, n=1002):')
print(f'{"With window":<25} F1={r_window["Mean F1"]:.4f} R@0.3={r_window["Recall@0.3"]:.4f} R@0.5={r_window["Recall@0.5"]:.4f}')
print(f'{"Without window":<25} F1={r_nowindow["Mean F1"]:.4f} R@0.3={r_nowindow["Recall@0.3"]:.4f} R@0.5={r_nowindow["Recall@0.5"]:.4f}')

# Save
with open('/kaggle/working/ablation_results.json', 'w') as f:
    json.dump({'obliqua_ablation': ablation_results,
               'rqa_window': {'with_window': r_window, 'without_window': r_nowindow,
                              'delta_f1': r_window['Mean F1'] - r_nowindow['Mean F1']}}, f, indent=2)
print('\nSaved: ablation_results.json')